In [1]:
# Install dependencies
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
#Load environment variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [4]:
def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": text
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": text
    }
    messages.append(assistant_message)

# Make a request
def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if system:
        params["system"] = system
        
    message = client.messages.create(**params)
    return message.content[0].text


In [5]:
import json

def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompt 
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON
    each representing task that requires Python, JSON, or a Regex to complete.
    
    Example output:
    ```json
    [
        {
            "task": "Description of the task",
            "formats": "python" or "json" or "regex"
        },
        ...additional
    ] 
    ``` 

    * Focus on tasks that can be solved writing a single Python function, a single JSON object,
    * Focus on tasks that do not require writing much code

    Please generate 3 objects
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [6]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [7]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:
    
    {test_case["task"]}

    * Respond only with Python, JSON, or a plain Regex
    * Do not add any comments or commentary or explanation
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output

In [27]:
import ast
import re


def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {test_case["task"]}
    Solution: {output}
    {"Solution Criteria you should use to evaluate the solution: " + test_case["solution_criteria"] if test_case.get("solution_criteria") else ""}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
    
def grade_syntax(response, test_case):
    format = test_case["formats"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    elif format == "regex":
        return validate_regex(response)
    else:
        raise ValueError(f"Unknown format: {format}")

In [28]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)
    # Combine model score and syntax score (you can adjust the weights as needed)
    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }


In [29]:
from statistics import mean


def run_eval(dataset):
    """Loads the dataset and calls run_test_case for each test case, then returns the results"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average Score: {average_score:.2f}")
        
    return results

In [16]:
with open("dataset.json") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

Average Score: 8.33


In [17]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\n\ndef parse_s3_uri(uri):\n    pattern = r'^s3://([a-z0-9.-]+)/(.+)$'\n    match = re.match(pattern, uri)\n    if match:\n        bucket = match.group(1)\n        key = match.group(2)\n        return {\"bucket\": bucket, \"key\": key}\n    return None\n\n# Test cases\nprint(parse_s3_uri(\"s3://my-bucket/path/to/object.txt\"))\nprint(parse_s3_uri(\"s3://my-bucket/object.txt\"))\nprint(parse_s3_uri(\"s3://bucket-123/file.zip\"))\n",
    "test_case": {
      "task": "Parse an AWS S3 bucket name and object key from an S3 URI (e.g., s3://my-bucket/path/to/object.txt)",
      "formats": "regex"
    },
    "score": 8.0,
    "reasoning": "The solution works for basic, well-formed S3 URIs but lacks robustness for real-world scenarios. The regex pattern is overly restrictive and the function doesn't validate against actual AWS S3 naming conventions. Error handling is minimal, making debugging difficult. For production use, the solution should be more permissive i

# Exercise

In [18]:
import json

def generate_dataset2():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompt 
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON
    each representing task that requires Python, JSON, or a Regex to complete.
    
    Example output:
    ```json
    [
        {
            "task": "Description of the task",
            "formats": "python" or "json" or "regex",
            "solution_criteria": "A concise description of what a correct solution should look like"
        },
        ...additional
    ] 
    ``` 

    * Focus on tasks that can be solved writing a single Python function, a single JSON object,
    * Focus on tasks that do not require writing much code

    Please generate 3 objects
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [19]:
dataset = generate_dataset2()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [30]:
with open("dataset.json") as f:
    dataset = json.load(f)
    
results = run_eval(dataset)

Average Score: 7.92


In [31]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\nimport json\n\ndef parse_and_validate_s3_bucket(uri):\n    # Parse the bucket name from S3 URI\n    match = re.match(r's3://([a-z0-9\\-\\.]+)(?:/.*)?$', uri)\n    \n    if not match:\n        return json.dumps({\"valid\": False, \"bucket\": None, \"error\": \"Invalid S3 URI format\"})\n    \n    bucket_name = match.group(1)\n    \n    # Validate bucket name against AWS naming conventions\n    # Rules: lowercase, hyphens/dots allowed, 3-63 characters\n    # Must start and end with lowercase letter or number\n    # Cannot be formatted as IP address\n    \n    if not (3 <= len(bucket_name) <= 63):\n        return json.dumps({\"valid\": False, \"bucket\": bucket_name, \"error\": \"Bucket name must be 3-63 characters\"})\n    \n    if not re.match(r'^[a-z0-9][a-z0-9\\-\\.]*[a-z0-9]$|^[a-z0-9]$', bucket_name):\n        return json.dumps({\"valid\": False, \"bucket\": bucket_name, \"error\": \"Invalid characters or invalid start/end\"})\n    \n    if re.match